# Exploratory Analysis
This notebook will be used to import raw datasets from Open Baltimore, profile the data, and determine the steps needed to clean and transform the data for analysis.
## Datasets and Source URLs
* NIBRS Homicide Data | https://data.baltimorecity.gov/maps/204beefe92a645d79fdf0969957bbdf8
* Vacant Building Notices | https://data.baltimorecity.gov/maps/691d65a5f85640e6aaa46930bd9dc102
* Median Household Income | https://data.baltimorecity.gov/maps/8613366cfbc7447a9efd9123604c65c1
* Population | https://data.baltimorecity.gov/maps/56d5b4e5480049e98315c2732aa48437
* CSAs (Community Statistical Areas) Reference | https://data.baltimorecity.gov/maps/9c96ae20e6cc41258015c2fd288716c4

In [1]:
# import dependencies
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
import csv

### Homicide Dataset
As the dependent variable, homicide data will be central to the project, so we will begin by profiling this dataset.

In [2]:
df_homicides = pd.read_csv('NIBRS_GroupA_Crime_Data_1908320685369947930.csv')

print(df_homicides.info())

<class 'pandas.DataFrame'>
RangeIndex: 952 entries, 0 to 951
Data columns (total 24 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowID            952 non-null    int64  
 1   CCNumber         952 non-null    int64  
 2   CrimeDateTime    952 non-null    str    
 3   CrimeCode        952 non-null    str    
 4   Description      952 non-null    str    
 5   Inside_Outside   949 non-null    str    
 6   Weapon           952 non-null    str    
 7   Shooting         0 non-null      float64
 8   Post             950 non-null    float64
 9   Gender           952 non-null    str    
 10  Age              912 non-null    float64
 11  Race             919 non-null    str    
 12  Ethnicity        790 non-null    str    
 13  Location         950 non-null    str    
 14  Old_District     475 non-null    str    
 15  New_District     494 non-null    str    
 16  Neighborhood     625 non-null    str    
 17  Latitude         951 non-nu

In [3]:
# remove unnecessary columns, rename columns for consistency

df_homicides = df_homicides.drop(columns=['RowID', 'CCNumber', 'CrimeCode', 'Description', 'Inside_Outside', 'Weapon', 'Shooting', 'Post', 'Gender', 'Age', 'Race', 'Ethnicity', 'Location', 'Old_District', 'New_District', 'Neighborhood', 'GeoLocation', 'PremiseType', 'Total_Incidents', 'x', 'y'])

df_homicides.rename(columns={'CrimeDateTime': 'date', 'Latitude': 'lat', 'Longitude': 'long'}, inplace=True)

In [4]:
# convert date column to datetime type & normalize time

df_homicides['date'] = pd.to_datetime(df_homicides['date'], format='mixed')
df_homicides['date'] = df_homicides['date'].dt.normalize()

In [14]:
# remove any entries missing lat & long values

df_homicides = df_homicides.dropna(subset=['lat', 'long'])
print(df_homicides.info())

<class 'pandas.DataFrame'>
Index: 951 entries, 0 to 951
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    951 non-null    datetime64[us]
 1   lat     951 non-null    float64       
 2   long    951 non-null    float64       
dtypes: datetime64[us](1), float64(2)
memory usage: 29.7 KB
None


In [15]:
print(df_homicides.head())

        date        lat       long
0 2026-06-20  39.351494 -76.683431
1 2026-06-20  39.317852 -76.611126
2 2026-06-24  39.353042 -76.609304
3 2026-06-30  39.303436 -76.638303
4 2026-06-27  39.281902 -76.683871


In [16]:
# determine date range present in dataset

print(df_homicides['date'].max())
print(df_homicides['date'].min())

2026-07-04 00:00:00
2022-01-01 00:00:00


### CSAs Dataset
Much of the transformation process will be dependent on the CSAs dataset for location data, so we will profile it next.
Note: homicide data will need to be joined via coordinates using the GeoSON version of the CSAs data. This step will be executed later when merging datasets.

In [7]:
df_csa = pd.read_csv('Community_Statistical_Areas_(CSAs)__Reference_Boundaries.csv')

print(df_csa.info())
print(df_csa.head())

<class 'pandas.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   FID        56 non-null     int64
 1   Community  56 non-null     str  
 2   Neigh      56 non-null     str  
 3   Link       56 non-null     str  
 4   CSA2020    55 non-null     str  
dtypes: int64(1), str(4)
memory usage: 2.3 KB
None
   FID                          Community  \
0    1      Allendale/Irvington/S. Hilton   
1    2    Beechfield/Ten Hills/West Hills   
2    3                      Belair-Edison   
3    4  Brooklyn/Curtis Bay/Hawkins Point   
4    5                             Canton   

                                               Neigh  \
0  Allendale, Carroll-South Hilton, Gwynns Falls,...   
1  Beechfield, Hunting Ridge, Ten Hills, Tremont,...   
2  Belair-Edison, Clifton Park, Four By Four, May...   
3  Brooklyn, Curtis Bay, Fairfield Area, Hawkins ...   
4                             Canton, Pat

In [8]:
# drop unnecessary columns & rename columns for consistency

df_csa = df_csa.drop(columns=['FID', 'Link'])
df_csa.rename(columns={'Community': 'csa_10', 'Neigh': 'neighborhood', 'CSA2020': 'csa_20'}, inplace=True)
print(df_csa.head())

                              csa_10  \
0      Allendale/Irvington/S. Hilton   
1    Beechfield/Ten Hills/West Hills   
2                      Belair-Edison   
3  Brooklyn/Curtis Bay/Hawkins Point   
4                             Canton   

                                        neighborhood  \
0  Allendale, Carroll-South Hilton, Gwynns Falls,...   
1  Beechfield, Hunting Ridge, Ten Hills, Tremont,...   
2  Belair-Edison, Clifton Park, Four By Four, May...   
3  Brooklyn, Curtis Bay, Fairfield Area, Hawkins ...   
4                             Canton, Patterson Park   

                              csa_20  
0      Allendale/Irvington/S. Hilton  
1    Beechfield/Ten Hills/West Hills  
2                      Belair-Edison  
3  Brooklyn/Curtis Bay/Hawkins Point  
4                             Canton  


### Vacant Buildings Dataset
This data represents the independent variable, so we will profile this dataset next.

In [9]:
df_vb = pd.read_csv('Vacant_Building_Notices.csv')

print(df_vb.info())

<class 'pandas.DataFrame'>
RangeIndex: 11624 entries, 0 to 11623
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   X                          11624 non-null  float64
 1   Y                          11624 non-null  float64
 2   OBJECTID                   11624 non-null  int64  
 3   NoticeNum                  11624 non-null  str    
 4   DateNotice                 11624 non-null  str    
 5   DateCancel                 0 non-null      float64
 6   DateAbate                  0 non-null      float64
 7   NT                         11624 non-null  str    
 8   OWNER_ABBR                 1095 non-null   str    
 9   HousingMarketTypology2023  11606 non-null  str    
 10  Council_District           11624 non-null  int64  
 11  Neighborhood               11624 non-null  str    
 12  BLOCKLOT                   11624 non-null  str    
 13  Address                    11624 non-null  str    
dtypes

In [10]:
# drop unnecessary columns & rename columns for consistency

df_vb = df_vb.drop(columns=['X', 'Y', 'OBJECTID', 'NoticeNum', 'DateCancel', 'DateAbate', 'NT', 'OWNER_ABBR', 'HousingMarketTypology2023', 'Council_District', 'BLOCKLOT', 'Address'])
df_vb.rename(columns={'DateNotice': 'date', 'Neighborhood': 'neighborhood'}, inplace=True)

In [11]:
# convert date column to datetime type & normalize time

df_vb['date'] = pd.to_datetime(df_vb['date'], format='mixed')
df_vb['date'] = df_vb['date'].dt.normalize()
print(df_vb.head())

                       date      neighborhood
0 2026-05-28 00:00:00+00:00  Carrollton Ridge
1 2010-07-17 00:00:00+00:00  Carrollton Ridge
2 2008-06-26 00:00:00+00:00  Carrollton Ridge
3 2022-12-05 00:00:00+00:00  Carrollton Ridge
4 2024-02-01 00:00:00+00:00  Carrollton Ridge


### Population Dataset
Here, we will extract the population control variable.

In [12]:
df_pop = pd.read_csv('Total_Population_-_Community_Statistical_Area.csv')

print(df_pop.info())

<class 'pandas.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   OBJECTID       54 non-null     int64  
 1   CSA2010        54 non-null     str    
 2   tpop10         54 non-null     int64  
 3   tpop20         54 non-null     int64  
 4   Shape__Area    54 non-null     float64
 5   Shape__Length  54 non-null     float64
dtypes: float64(2), int64(3), str(1)
memory usage: 2.7 KB
None


### Median Household Income Dataset
Here, we will extract the income control variable.

In [13]:
df_income = pd.read_csv('Median_Household_Income_-_Community_Statistical_Area.csv')

print(df_income.info())

<class 'pandas.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   OBJECTID       55 non-null     int64  
 1   CSA2010        55 non-null     str    
 2   mhhi10         55 non-null     float64
 3   mhhi11         55 non-null     float64
 4   mhhi12         55 non-null     float64
 5   mhhi13         55 non-null     float64
 6   mhhi14         55 non-null     float64
 7   mhhi15         55 non-null     float64
 8   mhhi16         55 non-null     float64
 9   mhhi17         55 non-null     float64
 10  mhhi18         55 non-null     float64
 11  mhhi19         55 non-null     float64
 12  CSA2020        55 non-null     str    
 13  mhhi20         55 non-null     float64
 14  mhhi21         55 non-null     float64
 15  mhhi22         55 non-null     float64
 16  mhhi23         55 non-null     float64
 17  Shape__Area    55 non-null     float64
 18  Shape__Length  55 non-n